# 06 - Banco de Dados

## Descrição

Este notebook documenta a camada de origem de dados do sistema: o banco MongoDB. O MongoDB armazena 10 colecoes com dados transacionais de um e-commerce ficticio, totalizando aproximadamente 150.000 documentos. Cada colecao possui schema de validacao (`$jsonSchema`), indices para performance e integridade, e um campo `updated_at` que serve como base para a carga incremental do pipeline.

---

## Colecoes e Cardinalidade

| # | Colecao | Documentos | Papel | Chave Primaria |
|---|---------|-----------:|-------|----------------|
| 1 | `clientes` | 15.000 | Cadastro de clientes | `id_cliente` (int) |
| 2 | `categorias` | 15.000 | Categorias de produtos | `id_categoria` (int) |
| 3 | `fornecedores` | 15.000 | Fornecedores dos produtos | `id_fornecedor` (int) |
| 4 | `produtos` | 15.000 | Catalogo de produtos | `id_produto` (int) |
| 5 | `cupons` | 15.000 | Cupons de desconto | `id_cupom` (int) |
| 6 | `pedidos` | 15.000 | Cabecalho dos pedidos | `id_pedido` (int) |
| 7 | `itens_pedido` | 15.000 | Itens (linhas) dos pedidos | `id_item` (int) |
| 8 | `pagamentos` | 15.000 | Pagamentos dos pedidos | `id_pagamento` (int) |
| 9 | `entregas` | 15.000 | Entregas dos pedidos | `id_entrega` (int) |
| 10 | `avaliacoes` | 15.000 | Avaliacoes de produtos | `id_avaliacao` (int) |

> **Nota**: Os dados sao sinteticos, gerados por scripts Python com Faker (locale pt_BR). Sementes fixas garantem determinismo. A janela temporal das datas varia entre 2023-01-01 e 2026-06-11 (~3,5 anos).

---

## Modelo de Relacionamentos (ER Diagram)

```mermaid
erDiagram
    clientes      ||--o{ pedidos      : "id_cliente"
    cupons        |o--o{ pedidos      : "id_cupom (opcional)"
    pedidos       ||--o{ itens_pedido : "id_pedido"
    produtos      ||--o{ itens_pedido : "id_produto"
    categorias    ||--o{ produtos     : "id_categoria"
    fornecedores  ||--o{ produtos     : "id_fornecedor"
    pedidos       ||--o{ pagamentos   : "id_pedido"
    pedidos       ||--o{ entregas     : "id_pedido"
    pedidos       ||--o{ avaliacoes   : "id_pedido"
    produtos      ||--o{ avaliacoes   : "id_produto"
    clientes      ||--o{ avaliacoes   : "id_cliente"
```

> **Nota**: Os relacionamentos sao por chave inteira (ex: `pedidos.id_cliente` referencia `clientes.id_cliente`), nao por `ObjectId`. As chaves estrangeiras sao sorteadas no intervalo `1..15000` via `random.randint(1, TOTAL)`, entao o MongoDB nao impoe integridade referencial — isso e responsabilidade da camada Silver.

---

## Dicionario de Dados

### 1. `clientes`

| Campo | Tipo BSON | Obrigatorio | Descricao |
|-------|-----------|:-----------:|-----------|
| `id_cliente` | int | sim | Identificador unico (1..15000) |
| `nome` | string | sim | Nome completo |
| `email` | string | sim | E-mail |
| `cpf` | string | sim | CPF (formatado com pontos e traco) |
| `telefone` | string | sim | Telefone (com DDD e +55) |
| `data_nascimento` | date | sim | Data de nascimento (idade 18..75) |
| `genero` | string | sim | `M`, `F` ou `Outro` (validado por enum) |
| `logradouro` | string | sim | Endereco |
| `cidade` | string | sim | Cidade |
| `estado` | string | sim | UF (sigla de 2 letras) |
| `cep` | string | sim | CEP (formatado com hifen) |
| `data_cadastro` | date | sim | Data de cadastro |
| `updated_at` | date | sim | Ultima atualizacao (>= data_cadastro) |

### 2. `categorias`

| Campo | Tipo BSON | Obrigatorio | Descricao |
|-------|-----------|:-----------:|-----------|
| `id_categoria` | int | sim | Identificador unico |
| `nome_categoria` | string | sim | Nome da categoria |
| `descricao` | string | sim | Descricao |
| `updated_at` | date | sim | Ultima atualizacao |

### 3. `fornecedores`

| Campo | Tipo BSON | Obrigatorio | Descricao |
|-------|-----------|:-----------:|-----------|
| `id_fornecedor` | int | sim | Identificador unico |
| `nome_fornecedor` | string | sim | Razao social |
| `cnpj` | string | sim | CNPJ (formatado) |
| `email` | string | sim | E-mail corporativo |
| `telefone` | string | sim | Telefone |
| `logradouro` | string | sim | Endereco |
| `cidade` | string | sim | Cidade |
| `estado` | string | sim | UF (sigla de 2 letras) |
| `cep` | string | sim | CEP |
| `updated_at` | date | sim | Ultima atualizacao |

### 4. `produtos`

| Campo | Tipo BSON | Obrigatorio | Descricao |
|-------|-----------|:-----------:|-----------|
| `id_produto` | int | sim | Identificador unico |
| `nome_produto` | string | sim | Nome (marca + adjetivo + palavra) |
| `descricao` | string | sim | Descricao |
| `preco` | double | sim | Preco de venda (aprox. 9,90..4999,90) |
| `estoque` | int | sim | Quantidade em estoque (0..500) |
| `id_categoria` | int | sim | FK → categorias.id_categoria |
| `id_fornecedor` | int | sim | FK → fornecedores.id_fornecedor |
| `marca` | string | sim | Marca |
| `peso_kg` | double | sim | Peso em kg (0,1..30,0) |
| `updated_at` | date | sim | Ultima atualizacao |

### 5. `cupons`

| Campo | Tipo BSON | Obrigatorio | Descricao |
|-------|-----------|:-----------:|-----------|
| `id_cupom` | int | sim | Identificador unico |
| `codigo` | string | sim | Codigo do cupom (unico) |
| `desconto_percentual` | int | sim | Percentual (5..50) |
| `valor_minimo` | int | sim | Valor minimo da compra |
| `data_validade` | date | sim | Validade |
| `ativo` | bool | sim | Cupom ativo |
| `updated_at` | date | sim | Ultima atualizacao |

### 6. `pedidos`

| Campo | Tipo BSON | Obrigatorio | Descricao |
|-------|-----------|:-----------:|-----------|
| `id_pedido` | int | sim | Identificador unico |
| `id_cliente` | int | sim | FK → clientes.id_cliente |
| `data_pedido` | date | sim | Data do pedido |
| `status` | string | sim | pendente, processando, enviado, entregue, cancelado |
| `valor_total` | double | sim | Valor total (20,00..3000,00) |
| `id_cupom` | int | null | FK → cupons.id_cupom (nulo em ~75% dos pedidos) |
| `updated_at` | date | sim | Ultima atualizacao (>= data_pedido) |

### 7. `itens_pedido`

| Campo | Tipo BSON | Obrigatorio | Descricao |
|-------|-----------|:-----------:|-----------|
| `id_item` | int | sim | Identificador unico |
| `id_pedido` | int | sim | FK → pedidos.id_pedido |
| `id_produto` | int | sim | FK → produtos.id_produto |
| `quantidade` | int | sim | Quantidade (1..10) |
| `valor_unitario` | double | sim | Valor unitario |
| `desconto_percentual` | double | sim | Desconto aplicado (%) |
| `subtotal` | double | sim | `quantidade * valor_unitario * (1 - desconto)` |
| `updated_at` | date | sim | Ultima atualizacao |

### 8. `pagamentos`

| Campo | Tipo BSON | Obrigatorio | Descricao |
|-------|-----------|:-----------:|-----------|
| `id_pagamento` | int | sim | Identificador unico |
| `id_pedido` | int | sim | FK → pedidos.id_pedido |
| `forma_pagamento` | string | sim | cartao_credito, cartao_debito, pix, boleto |
| `status_pagamento` | string | sim | aprovado, pendente, recusado, estornado |
| `valor` | double | sim | Valor pago |
| `data_pagamento` | date | sim | Data do pagamento |
| `parcelas` | int | sim | Numero de parcelas (1..12) |
| `updated_at` | date | sim | Ultima atualizacao |

### 9. `entregas`

| Campo | Tipo BSON | Obrigatorio | Descricao |
|-------|-----------|:-----------:|-----------|
| `id_entrega` | int | sim | Identificador unico |
| `id_pedido` | int | sim | FK → pedidos.id_pedido |
| `status_entrega` | string | sim | pendente, em_transito, entregue, devolvido |
| `data_envio` | date | sim | Data de envio |
| `data_entrega_prevista` | date | sim | Previsao de entrega |
| `data_entrega_real` | date | null | Entrega efetiva (nulo se nao entregue) |
| `transportadora` | string | sim | Correios, JadLog, Total Express, Loggi, Azul Cargo, Latam Cargo, DHL, FedEx |
| `codigo_rastreio` | string | sim | Codigo de rastreio (formato BR: XX123456789BR) |
| `updated_at` | date | sim | Ultima atualizacao |

### 10. `avaliacoes`

| Campo | Tipo BSON | Obrigatorio | Descricao |
|-------|-----------|:-----------:|-----------|
| `id_avaliacao` | int | sim | Identificador unico |
| `id_pedido` | int | sim | FK → pedidos.id_pedido |
| `id_cliente` | int | sim | FK → clientes.id_cliente |
| `id_produto` | int | sim | FK → produtos.id_produto |
| `nota` | int | sim | Nota de 1 a 5 |
| `comentario` | string | sim | Comentario textual |
| `data_avaliacao` | date | sim | Data da avaliacao |
| `updated_at` | date | sim | Ultima atualizacao |

---

## Validacao de Schema (`$jsonSchema`)

O MongoDB suporta validacao de schema por colecao via `$jsonSchema`. Os schemas definem:

- **Campos obrigatorios** via `required` (ex: `id_cliente`, `nome`, `updated_at`)
- **Tipos BSON** por campo (ex: `int`, `string`, `date`, `bool`)
- **Domnios fechados** via `enum` (ex: `genero` ∈ {M, F, Outro})
- **Restricoes de formato** (ex: `estado` com `minLength`/`maxLength` = 2)

### Exemplo: Schema `clientes`

```json
{
  "$jsonSchema": {
    "bsonType": "object",
    "required": ["id_cliente", "nome", "email", "cpf", "telefone", "data_nascimento", "genero", "logradouro", "cidade", "estado", "cep", "data_cadastro", "updated_at"],
    "properties": {
      "id_cliente": { "bsonType": ["int", "long"] },
      "nome": { "bsonType": "string" },
      "email": { "bsonType": "string" },
      "cpf": { "bsonType": "string" },
      "telefone": { "bsonType": "string" },
      "data_nascimento": { "bsonType": "date" },
      "genero": { "enum": ["M", "F", "Outro"] },
      "logradouro": { "bsonType": "string" },
      "cidade": { "bsonType": "string" },
      "estado": { "bsonType": "string", "minLength": 2, "maxLength": 2 },
      "cep": { "bsonType": "string" },
      "data_cadastro": { "bsonType": "date" },
      "updated_at": { "bsonType": "date" }
    }
  }
}
```

### Exemplo: Schema `pedidos` (com campo opcional)

```json
{
  "$jsonSchema": {
    "bsonType": "object",
    "required": ["id_pedido", "id_cliente", "data_pedido", "status", "valor_total", "updated_at"],
    "properties": {
      "id_pedido": { "bsonType": ["int", "long"] },
      "id_cliente": { "bsonType": ["int", "long"] },
      "data_pedido": { "bsonType": "date" },
      "status": { "enum": ["pendente", "processando", "enviado", "entregue", "cancelado"] },
      "valor_total": { "bsonType": ["double"] },
      "id_cupom": { "bsonType": ["int", "long", "null"] },
      "updated_at": { "bsonType": "date" }
    }
  }
}
```

> **Referencia**: `mongodb/schemas/clientes.schema.json`, `mongodb/schemas/pedidos.schema.json`

---

## Indices

Todos os indices sao criados automaticamente pelo script `carregar_mongo.py` na carga inicial:

| Colecao | Indice | Tipo | Finalidade |
|---------|--------|------|------------|
| Todas | `id_*` (PK) | Unico, Ascendente | Busca por chave primaria, garantia de unicidade |
| Todas | `updated_at` | Ascendente | Carga incremental (filtro por timestamp) |
| `pedidos` | `id_cliente` | Ascendente | FK para clientes (joins, validacao) |
| `pedidos` | `id_cupom` | Ascendente | FK para cupons (joins, validacao) |
| `produtos` | `id_categoria` | Ascendente | FK para categorias |
| `produtos` | `id_fornecedor` | Ascendente | FK para fornecedores |
| `itens_pedido` | `id_pedido` | Ascendente | FK para pedidos |
| `itens_pedido` | `id_produto` | Ascendente | FK para produtos |
| `pagamentos` | `id_pedido` | Ascendente | FK para pedidos |
| `entregas` | `id_pedido` | Ascendente | FK para pedidos |
| `avaliacoes` | `id_pedido` | Ascendente | FK para pedidos |
| `avaliacoes` | `id_cliente` | Ascendente | FK para clientes |
| `avaliacoes` | `id_produto` | Ascendente | FK para produtos |

> **Nota**: O indice em `updated_at` e essencial para a carga incremental — permite que a query `find({updated_at: {$gte: checkpoint}})` use o indice em vez de fazer collection scan.

---

## Estrategia de Carga Incremental

Todas as colecoes possuem o campo `updated_at` (data/hora da ultima alteracao do documento). Esse campo e o controle para a carga incremental do pipeline.

### Como funciona:

1. O pipeline guarda um **checkpoint** — o maior `updated_at` ja processado por colecao
2. Na ingestao seguinte, busca apenas documentos com `updated_at` maior que o checkpoint (com overlap de 24h)
3. Atualiza o checkpoint para o novo maximo

### Exemplo:

```text
Checkpoint anterior: 2026-06-10T00:00:00Z
Overlap: 24 horas
Filtro: updated_at >= 2026-06-09T00:00:00Z
Novos documentos: 150 com updated_at entre 2026-06-09 e 2026-06-13
Novo checkpoint: 2026-06-13T12:30:00Z
```

> **Referencia**: `dags/lib/mongodb_landing.py` (funcoes `build_incremental_filter`, `parse_checkpoint`, `format_checkpoint`)

---

## Exemplos de Documentos (apos conversao para Extended JSON)

### Cliente

```json
{
  "_id": { "$oid": "665000000000000000000001" },
  "id_cliente": { "$numberInt": "1" },
  "nome": "Ana Beatriz da Rocha",
  "email": "ana.rocha@example.com",
  "cpf": "123.456.789-00",
  "telefone": "+55 11 99999-0001",
  "data_nascimento": { "$date": { "$numberLong": "641174400000" } },
  "genero": "F",
  "logradouro": "Rua das Acacias, 100",
  "cidade": "Sao Paulo",
  "estado": "SP",
  "cep": "01234-567",
  "data_cadastro": { "$date": { "$numberLong": "1678838400000" } },
  "updated_at": { "$date": { "$numberLong": "1722374400000" } }
}
```

### Pedido (com cupom)

```json
{
  "_id": { "$oid": "665000000000000000000042" },
  "id_pedido": { "$numberInt": "42" },
  "id_cliente": { "$numberInt": "8123" },
  "data_pedido": { "$date": { "$numberLong": "1716163200000" } },
  "status": "entregue",
  "valor_total": { "$numberDouble": "459.90" },
  "id_cupom": { "$numberInt": "311" },
  "updated_at": { "$date": { "$numberLong": "1717372800000" } }
}
```

### Pedido (sem cupom)

```json
{
  "_id": { "$oid": "665000000000000000000043" },
  "id_pedido": { "$numberInt": "43" },
  "id_cliente": { "$numberInt": "277" },
  "data_pedido": { "$date": { "$numberLong": "1736448000000" } },
  "status": "processando",
  "valor_total": { "$numberDouble": "1280.00" },
  "id_cupom": null,
  "updated_at": { "$date": { "$numberLong": "1736611200000" } }
}
```
